# Notebook 09 — Evolutionary Game Theory

**Module:** 06 — Genetics and Evolution  
**Notebook:** 09 of 12  
**Prerequisites:** NB04 (selection models), Module 05 NB08 (fitness models)  
**Time estimate:** 75 minutes

> Evolutionary game theory explains why frequency-dependent selection maintains
> diversity — why both hawks and doves coexist, why altruism can evolve, and why
> some virulence strategies are evolutionarily stable. It also connects directly to
> the agent-based modeling module (Module 15).

---
## Step 1 — Motivation

Classical selection assumes fitness is constant. But in many biological situations,
the fitness of a strategy depends on what everyone else is doing — the payoff from
being aggressive depends on how many other aggressive individuals you face. This is
**frequency-dependent selection**, and game theory formalises it. The key concept,
the **Evolutionarily Stable Strategy (ESS)**, predicts which strategy (or mixture)
a population converges to and cannot be invaded by.

---
## Step 3 — Biological Background

### Classic Games in Biology

**Hawk-Dove (Maynard Smith & Price 1973):**
- Hawks escalate conflict until winning or injured
- Doves display but retreat if opponent escalates
- ESS: mixed strategy with p* = V/C if V < C (benefit-to-cost ratio)

**Prisoner's Dilemma:**
- Cooperate or Defect; defection always pays more in a single round
- But if the game is iterated, Tit-for-Tat can invade
- Applies to: immune evasion, cooperative hunting, biofilm formation

**Rock-Paper-Scissors (non-transitive competition):**
- Three strategies each beat one and lose to one
- No pure ESS; cycling population frequencies
- Real example: side-blotched lizard (Uta stansburiana) colour morphs

### Evolutionarily Stable Strategy (ESS) Definition

Strategy I is an ESS if:
1. W(I, I) > W(J, I) for all J ≠ I, OR
2. W(I, I) = W(J, I) AND W(I, J) > W(J, J)

Where W(X, Y) = payoff to strategy X when playing against strategy Y.

### Replicator Dynamics

How strategy frequencies change over time given a payoff matrix:
$$\dot{x}_i = x_i \left[ (A\mathbf{x})_i - \mathbf{x}^T A \mathbf{x} \right]$$
where x = frequency vector, A = payoff matrix, (Ax)_i = fitness of strategy i.

---
## Step 4 — Mathematical Explanation

**Hawk-Dove payoff matrix:**

|        | Hawk | Dove |
|--------|------|------|
| **Hawk** | (V−C)/2 | V |
| **Dove** | 0 | V/2 |

Where V = value of resource, C = cost of injury.

**ESS condition for Hawk-Dove (V < C):**
$$p^* = \frac{V}{C}$$

At p* hawks, average fitness of hawks = average fitness of doves.
Increasing p above p* makes hawks rarer (dove invasion succeeds).

**Replicator equation for two strategies (x = freq of Hawk):**
$$\dot{x} = x(1-x)[W_{H} - W_{D}]$$
where W_H = average payoff to Hawk, W_D = average payoff to Dove.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# Cell 6.1 — Hawk-Dove ESS calculation
def hawk_dove_ess(V: float, C: float) -> dict:
    """
    Compute the ESS for the Hawk-Dove game.

    V: value of contested resource
    C: cost of injury in escalated contest
    Returns: ESS type and mixed strategy frequency
    """
    if V >= C:
        # Hawk dominates: pure Hawk strategy
        return {'ess': 'Pure Hawk', 'p_hawk': 1.0,
                'note': 'Benefits exceed costs — always escalate'}
    else:
        p_star = V / C
        return {'ess': 'Mixed', 'p_hawk': p_star,
                'note': f'{100*p_star:.1f}% Hawks, {100*(1-p_star):.1f}% Doves at equilibrium'}

# Biological examples
scenarios = [
    ('Sticklebacks (low cost)', 10, 20),
    ('Deer antler fights (high cost)', 5, 50),
    ('Bacteria (resource > injury)', 10, 5),
]
for name, V, C in scenarios:
    result = hawk_dove_ess(V, C)
    print(f"{name}: V={V}, C={C}")
    print(f"  ESS = {result['ess']}, p(Hawk)* = {result['p_hawk']:.3f}")
    print(f"  {result['note']}")

In [ ]:
# Cell 6.2 — Replicator dynamics for n-strategy games
def replicator_dynamics(payoff_matrix: np.ndarray, x0: np.ndarray,
                         t_max: float = 50.0, n_points: int = 1000) -> tuple:
    """
    Simulate replicator dynamics for an n-strategy evolutionary game.

    payoff_matrix: (n,n) matrix; A[i,j] = payoff to i when j is played
    x0: initial strategy frequencies (must sum to 1)
    Returns: (t, x) where x has shape (n_points, n)
    """
    A = payoff_matrix
    n = len(x0)

    def rhs(t, x):
        # Fitness of each strategy: (A @ x)[i]
        f = A @ x
        # Mean fitness
        f_bar = x @ f
        # Replicator equation
        dxdt = x * (f - f_bar)
        return dxdt

    t_span = (0, t_max)
    t_eval = np.linspace(0, t_max, n_points)

    sol = solve_ivp(rhs, t_span, x0, t_eval=t_eval, method='RK45',
                    rtol=1e-8, atol=1e-10)
    return sol.t, sol.y.T  # x[time, strategy]


# Hawk-Dove replicator dynamics
V, C = 5, 50
A_hd = np.array([
    [(V-C)/2, V],   # Hawk vs (Hawk, Dove)
    [0,       V/2]  # Dove vs (Hawk, Dove)
], dtype=float)

p_star_hd = hawk_dove_ess(V, C)['p_hawk']

results_hd = []
for p_init in [0.05, 0.3, 0.6, 0.95]:
    t, x = replicator_dynamics(A_hd, [p_init, 1-p_init], t_max=100.0)
    results_hd.append((p_init, t, x))
print(f"Hawk-Dove (V={V}, C={C}): ESS p(Hawk)* = {p_star_hd:.2f}")
for p0, t, x in results_hd:
    print(f"  p0={p0:.2f} → p(Hawk) at t=100: {x[-1, 0]:.4f}")

In [ ]:
# Cell 6.3 — Rock-Paper-Scissors: cycling dynamics
A_rps = np.array([
    [0,  1, -1],   # Rock: beats Scissors, loses to Paper
    [-1, 0,  1],   # Paper: beats Rock, loses to Scissors
    [1, -1,  0],   # Scissors: beats Paper, loses to Rock
], dtype=float)

t_rps, x_rps = replicator_dynamics(A_rps, [0.8, 0.1, 0.1], t_max=200.0, n_points=2000)
print("Rock-Paper-Scissors: no pure ESS, strategy frequencies cycle")
print(f"  Final frequencies (t=200): Rock={x_rps[-1,0]:.4f}, Paper={x_rps[-1,1]:.4f}, Scissors={x_rps[-1,2]:.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Hawk-Dove replicator trajectories + RPS cycling
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Hawk-Dove frequency over time
ax = axes[0]
colors = ['tomato', 'orange', 'steelblue', 'seagreen']
for (p0, t, x), color in zip(results_hd, colors):
    ax.plot(t, x[:, 0], color=color, lw=2, label=f'p₀(Hawk)={p0:.2f}')
ax.axhline(p_star_hd, color='black', ls='--', lw=1.5, label=f'ESS p*={p_star_hd:.2f}')
ax.set_xlabel('Time')
ax.set_ylabel('Frequency of Hawk strategy')
ax.set_title(f'Hawk-Dove replicator dynamics (V={V}, C={C}):\nall initial conditions converge to ESS')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)

# Panel 2: Rock-Paper-Scissors cycling (simplex triangle)
ax = axes[1]
# Plot on simplex via barycentric coordinates
# Map (r, p, s) → 2D: x = 0.5*r + s, y = (√3/2)*r  (simplex projection)
def to_simplex(freqs):
    r, p, s = freqs[:, 0], freqs[:, 1], freqs[:, 2]
    x2d = 0.5 * r + s
    y2d = (np.sqrt(3) / 2) * r
    return x2d, y2d

sx, sy = to_simplex(x_rps)
ax.plot(sx, sy, color='steelblue', lw=1.2, alpha=0.7)
ax.scatter(sx[0], sy[0], color='tomato', s=60, zorder=5, label='Start')
ax.scatter(sx[-1], sy[-1], color='seagreen', s=60, zorder=5, label='End')

# Simplex triangle
corners = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3)/2], [0, 0]])
ax.plot(corners[:, 0], corners[:, 1], 'k-', lw=1.5)
offset = 0.04
ax.text(0 - offset, 0 - offset, 'Scissors', ha='right', fontsize=9)
ax.text(1 + offset, 0 - offset, 'Paper', ha='left', fontsize=9)
ax.text(0.5, np.sqrt(3)/2 + offset, 'Rock', ha='center', fontsize=9)

# Centre point (equal mix)
cx, cy = to_simplex(np.array([[1/3, 1/3, 1/3]]))
ax.scatter(cx, cy, color='gray', s=40, marker='x', zorder=5, label='ESS (1/3 each)')
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Rock-Paper-Scissors cycling:\nno stable ESS, trajectory orbits centre')
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `replicator_dynamics(A, x0, t_max)` from scratch. Verify that for the
   Hawk-Dove game with V=10, C=20, all initial conditions converge to p*=0.5.
2. Prisoner's Dilemma: payoff matrix is [[R,S],[T,P]] where T>R>P>S.
   With values T=5, R=3, P=1, S=0, show that Defect is the dominant strategy
   (pure ESS). Now add a third strategy: Tit-for-Tat (cooperate first, then
   mirror). Modify the matrix and re-run replicator dynamics.
3. In the side-blotched lizard (RPS dynamics with three throat-colour morphs),
   why does the population cycle instead of converging? What would need to be true
   for it to converge instead?
4. Kin selection (Hamilton's rule): altruistic behaviour evolves when r·B > C,
   where r = relatedness, B = benefit to recipient, C = cost to actor. For full
   siblings (r=0.5), what is the minimum B/C ratio required for altruism to evolve?

---
## Quiz — Active Recall

1. What is frequency-dependent selection? Give a biological example.
2. Define ESS. When is a mixed strategy an ESS?
3. For Hawk-Dove with V=10, C=20, what fraction of the population is Hawk at ESS?
4. State the replicator equation in words.
5. Why is Defect the dominant strategy in a single-round Prisoner's Dilemma
   even though mutual cooperation (R,R) beats mutual defection (P,P)?

---
## Papers Referenced

- Maynard Smith & Price (1973). DOI: 10.1038/246015a0

---
## Reflection

**Date completed:** ____________________

> *[SARS-CoV-2 variants compete within a host and across a population. How would
> you frame variant competition as an evolutionary game? What would the payoff matrix
> entries represent?]*

---
*Next: `10_convergent_and_divergent_evolution.ipynb`*